# Ensemble Methods

## Learning Objectives
1. Understand how combining multiple weak learners reduces variance (bagging) or bias (boosting).
2. Implement majority voting and soft voting ensemble from scratch with NumPy.
3. Compare BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier and VotingClassifier on synthetic data.
4. Build a stacking ensemble with a meta-learner and analyse feature importances from Random Forest and XGBoost.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


## Level 1: Majority Voting + Soft Voting (NumPy)

In [ ]:
# Three toy classifiers represented as probability arrays
np.random.seed(42)
n_samples = 200

# Simulate predictions from three classifiers on a binary problem
# prob shape: (n_samples, n_classes=2)
def toy_probs(noise=0.1):
    """Generate random but biased probability predictions."""
    true_labels = np.random.randint(0, 2, n_samples)
    probs = np.zeros((n_samples, 2))
    for i, label in enumerate(true_labels):
        correct_prob = np.clip(0.7 + np.random.normal(0, noise), 0, 1)
        probs[i, label] = correct_prob
        probs[i, 1 - label] = 1 - correct_prob
    return probs, true_labels


p1, true_y = toy_probs(noise=0.15)
p2, _ = toy_probs(noise=0.20)
p3, _ = toy_probs(noise=0.10)


def majority_vote(probs_list):
    """Hard majority vote: take argmax per classifier, then vote."""
    votes = np.stack([np.argmax(p, axis=1) for p in probs_list], axis=1)
    # shape: (n_samples, n_classifiers)
    ensemble_pred = np.apply_along_axis(
        lambda row: np.bincount(row, minlength=2).argmax(), axis=1, arr=votes
    )
    return ensemble_pred


def soft_vote(probs_list):
    """Average probabilities, then argmax."""
    avg_probs = np.mean(np.stack(probs_list, axis=0), axis=0)
    return np.argmax(avg_probs, axis=1)


hard_preds = majority_vote([p1, p2, p3])
soft_preds = soft_vote([p1, p2, p3])
single_preds = np.argmax(p1, axis=1)

print(f"Single classifier accuracy  : {accuracy_score(true_y, single_preds):.4f}")
print(f"Majority (hard) vote accuracy: {accuracy_score(true_y, hard_preds):.4f}")
print(f"Soft vote accuracy           : {accuracy_score(true_y, soft_preds):.4f}")


## Level 2: Bagging, Boosting, Voting with sklearn

In [ ]:
from sklearn.ensemble import (
    BaggingClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, VotingClassifier, RandomForestClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings("ignore")

# Generate synthetic dataset
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Base estimator for bagging/boosting
base_dt = DecisionTreeClassifier(max_depth=3, random_state=42)

# Define ensemble classifiers
ensembles = {
    "DecisionTree (base)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "BaggingClassifier": BaggingClassifier(
        estimator=base_dt, n_estimators=50, random_state=42, n_jobs=-1
    ),
    "AdaBoostClassifier": AdaBoostClassifier(
        estimator=base_dt, n_estimators=100, learning_rate=0.1, random_state=42
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
    ),
    "VotingClassifier (soft)": VotingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
            ("gb", GradientBoostingClassifier(n_estimators=50, random_state=42)),
            ("lr", LogisticRegression(max_iter=500, random_state=42)),
        ],
        voting="soft",
    ),
}

results = {}
for name, clf in ensembles.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f"{name:<35}: accuracy = {acc:.4f}")

best = max(results, key=results.get)
print(f"
Best ensemble: {best} ({results[best]:.4f})")


## Real-World Example 1: Random Forest Feature Importance

In [ ]:
# Feature importance reveals which predictors drive RF decisions.
# Two measures: impurity-based (fast, built-in) and permutation (reliable, slower).

from sklearn.inspection import permutation_importance

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Built-in impurity-based importance
impurity_importance = rf.feature_importances_
feature_names = [f"feat_{i:02d}" for i in range(X.shape[1])]

# Sort features by importance
sorted_idx = np.argsort(impurity_importance)[::-1]

print("Top-10 features by impurity importance:")
for rank, idx in enumerate(sorted_idx[:10], 1):
    print(f"  {rank:2d}. {feature_names[idx]}: {impurity_importance[idx]:.4f}")

# Permutation importance (more reliable — measures actual prediction impact)
perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
perm_sorted_idx = perm_result.importances_mean.argsort()[::-1]

print("
Top-10 features by permutation importance:")
for rank, idx in enumerate(perm_sorted_idx[:10], 1):
    print(f"  {rank:2d}. {feature_names[idx]}: {perm_result.importances_mean[idx]:.4f} "
          f"(+/- {perm_result.importances_std[idx]:.4f})")


## Real-World Example 2: XGBoost with Early Stopping + Feature Importance

In [ ]:
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — showing sklearn GBM equivalent")

if HAS_XGB:
    from sklearn.model_selection import train_test_split as tts

    X_tr, X_val, y_tr, y_val = tts(X_train, y_train, test_size=0.15, random_state=42)

    xgb_clf = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="logloss",
        early_stopping_rounds=20,
        random_state=42,
        verbosity=0,
    )
    xgb_clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    xgb_acc = accuracy_score(y_test, xgb_clf.predict(X_test))
    print(f"XGBoost accuracy: {xgb_acc:.4f}  (best iteration: {xgb_clf.best_iteration})")

    # Feature importance from XGBoost (gain-based)
    gain_importance = xgb_clf.feature_importances_
    top5_idx = gain_importance.argsort()[::-1][:5]
    print("Top-5 features by XGBoost gain:")
    for idx in top5_idx:
        print(f"  {feature_names[idx]}: {gain_importance[idx]:.4f}")
else:
    # Fallback using sklearn's GBM
    gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
    gb.fit(X_train, y_train)
    print(f"GBM accuracy: {accuracy_score(y_test, gb.predict(X_test)):.4f}")


## Real-World Example 3: Stacking Ensemble with Meta-Learner

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Stacking: base learners produce out-of-fold predictions;
# meta-learner (LogisticRegression) learns how to combine them.

base_learners = [
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)),
    ("gb", GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)),
    ("svc", make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))),
]

meta_learner = LogisticRegression(C=1.0, max_iter=500, random_state=42)

stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,                  # 5-fold cross-val to produce meta-features
    stack_method="predict_proba",  # use probabilities, not hard labels
    n_jobs=-1,
    passthrough=False,     # meta-learner only sees base predictions, not raw features
)

stacking_clf.fit(X_train, y_train)
stacking_acc = accuracy_score(y_test, stacking_clf.predict(X_test))
print(f"Stacking ensemble accuracy: {stacking_acc:.4f}")

# Compare to individual base learners
for name, clf in base_learners:
    clf_copy = clf.__class__(**clf.get_params()) if hasattr(clf, 'get_params') else clf
    # Re-use already-fitted base learners from stacking for fair comparison
print("
Summary: stacking pools diverse models; works best when base errors are uncorrelated.")
print(f"Stacking accuracy ({stacking_acc:.4f}) vs best single ({max(results.values()):.4f})")
